# Week 4: Hilbert space numerically

**What we do.** We represent functions on `[0, 1]` by their samples on a fine grid and compute inner products, norms, Gram matrices, Riesz bounds, an orthonormal basis (Gram–Schmidt) and orthogonal projections. The example family is the piecewise linear "hat" functions (splines of order 1) from the lecture notes.

**What to expect.** Hat functions overlap, so they are not orthogonal: their Gram matrix is tridiagonal, not the identity. They are nevertheless a Riesz basis (the lower Riesz bound is positive). Gram–Schmidt turns them into an orthonormal family spanning the same space, and the projection gives the best piecewise linear approximation of a function.

**How to read the code.** A sampled function is a 1D array; a family of functions is a 2D array with one function per row. The inner product `<f, g> = h Σ f[n] conj(g[n])` (with grid step `h`) approximates the integral `∫ f conj(g)`. All functions come from `wavelets_lab.hilbert`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import hilbert

## Inner products of sampled functions

**What we do.** We check numerically two facts from the lecture: `sin(2πx)` and `cos(2πx)` are orthogonal on `[0, 1]`, and `||sin(2πx)||² = 1/2`.

**What to expect.** The inner product is (almost) 0 and the squared norm is (almost) 0.5; the small deviations come from replacing the integral by a sum.

**How to read the code.** `grid` has 501 points from 0 to 1 with step `step = 0.002`. `hilbert.inner_product(f, g, step)` returns a complex number (functions may be complex); `hilbert.norm(f, step)` returns `sqrt(<f, f>)`.

In [ ]:
grid = np.linspace(0.0, 1.0, 501)
step = grid[1] - grid[0]
sine, cosine = np.sin(2 * np.pi * grid), np.cos(2 * np.pi * grid)
hilbert.inner_product(sine, cosine, step), hilbert.norm(sine, step) ** 2

## Hat functions: a Riesz basis that is not orthogonal

**What we do.** We build six hat functions with knots `0, 0.2, …, 1`, plot them and print their Gram matrix `G[i, j] = <e_j, e_i>`.

**What to expect.** Each hat equals 1 at its own knot and 0 at the others. The Gram matrix has `h/3 ≈ 0.133` on the diagonal (`h/6 ≈ 0.067` for the two boundary half-hats), `h/6 ≈ 0.033` next to the diagonal (neighbours overlap) and zeros elsewhere (`h = 0.2` is the distance of the knots).

**How to read the code.** `hilbert.hat_functions(grid, knots)` returns an array of shape `(6, 501)`; the loop plots each row. `np.round(..., 3)` rounds for printing and `.real` drops the zero imaginary part.

In [ ]:
knots = np.linspace(0.0, 1.0, 6)
hats = hilbert.hat_functions(grid, knots)
plt.figure(figsize=(8, 3))
for index, row in enumerate(hats):
    plt.plot(grid, row, label=f"e_{index}")
plt.xlabel("x")
plt.ylabel("e_k(x)")
plt.title("hat functions")
plt.legend(ncol=6)
plt.show()
np.round(hilbert.gram_matrix(hats, step).real, 3)

## Riesz bounds

**What we do.** We compute the optimal Riesz bounds `A ≤ B` of the hat family, i.e. the smallest and largest eigenvalue of the Gram matrix, and compare them with an orthonormal family.

**What to expect.** For the hats `0 < A < B`: they are linearly independent (a Riesz basis of their span) but not orthonormal. For the rows of the identity matrix (the standard basis of `ℓ²`, step 1) both bounds equal 1.

**How to read the code.** `hilbert.riesz_bounds(family, step)` returns the pair `(A, B)`; `np.eye(4)` is the 4×4 identity matrix.

In [ ]:
hilbert.riesz_bounds(hats, step), hilbert.riesz_bounds(np.eye(4))

## Gram–Schmidt

**What we do.** We orthonormalize the hats with the Gram–Schmidt process and check the result.

**What to expect.** Six functions that are still piecewise linear with the same knots (they span the same space) but are orthonormal; the first one is a rescaled hat, the later ones have negative parts that cancel the overlaps. `is_orthonormal` returns `True` for the new family and `False` for the hats.

**How to read the code.** `hilbert.gram_schmidt(hats, step)` returns the new family (one function per row); `.real` is used for plotting because the result has a complex-compatible type.

In [ ]:
basis = hilbert.gram_schmidt(hats, step)
plt.figure(figsize=(8, 3))
for index, row in enumerate(basis):
    plt.plot(grid, row.real, label=f"u_{index}")
plt.xlabel("x")
plt.ylabel("u_k(x)")
plt.title("orthonormalized hat functions")
plt.legend(ncol=6)
plt.show()
hilbert.is_orthonormal(basis, step), hilbert.is_orthonormal(hats, step)

## Projection onto the spline space

**What we do.** We project `f(x) = sin(2πx) + x²` orthogonally onto the span of the hats, i.e. we find the closest piecewise linear function with knots `0, 0.2, …, 1`.

**What to expect.** A broken line close to `f`. It is **not** the interpolant through the values at the knots: the orthogonal projection minimizes the `L²` error over the whole interval. The residual `f - Pf` is orthogonal to every hat, so the printed inner products are about 0.

**How to read the code.** `hilbert.project(f, hats, step)` solves the normal equations `G c = (<f, e_i>)` and returns `Σ c_k e_k`. The last line returns the `L²` error `||f - Pf||` and the inner products of the residual with all hats.

In [ ]:
f = np.sin(2 * np.pi * grid) + grid**2
projection = hilbert.project(f, hats, step).real
plt.figure(figsize=(8, 4))
plt.plot(grid, f, label="f")
plt.plot(grid, projection, label="projection Pf")
plt.plot(knots, np.interp(knots, grid, f), "o", label="f at the knots")
plt.xlabel("x")
plt.ylabel("value")
plt.legend()
plt.show()
residual = f - projection
(
    hilbert.norm(residual, step),
    np.round([hilbert.inner_product(residual, e, step).real for e in hats], 10),
)

## Exercise: more knots

**What we do.** We repeat the projection with more and more knots and record the approximation error and the Riesz bounds.

**What to expect.** The error decreases roughly like `h²` (four times smaller when the number of intervals doubles), the typical rate for piecewise linear approximation of a smooth function. The Riesz bounds shrink proportionally to `h`, because each hat gets narrower; their ratio `B / A` stays bounded (between about 3 and 4, printed below), which means the family stays "uniformly" a Riesz basis.

**How to read the code.** The loop builds families with `n + 1` knots; the results are plotted on logarithmic axes (`loglog`), on which a power law `h^p` is a straight line of slope `p`.

In [ ]:
intervals = [5, 10, 20, 40, 80]
errors, ratios = [], []
for n in intervals:
    family = hilbert.hat_functions(grid, np.linspace(0.0, 1.0, n + 1))
    errors.append(hilbert.norm(f - hilbert.project(f, family, step), step))
    lower, upper = hilbert.riesz_bounds(family, step)
    ratios.append(upper / lower)
spacing = 1 / np.array(intervals)
plt.figure(figsize=(6, 4))
plt.loglog(spacing, errors, "o-", label="error ||f - Pf||")
plt.loglog(spacing, spacing**2, "k--", label="slope 2 (h^2)")
plt.xlabel("knot spacing h")
plt.ylabel("L2 error")
plt.legend()
plt.show()
np.round(ratios, 3)

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.hilbert` (`src/wavelets_lab/hilbert.py`). Functions marked *used internally* are called by `project`.

**How to read the code.** In all functions a *family* is a 2D array with one sampled function per row and `step` is the grid step `h` of the Riemann sum.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.linspace(a, b, n)` | `n` equally spaced points | numbers → 1D array |
| `np.interp(x, xp, fp)` | linear interpolation of the data `(xp, fp)` at `x` | arrays → array |
| `np.eye(n)` | identity matrix | size → 2D array |
| `np.round(a, d)` | rounds to `d` decimals | array → array |
| `plt.loglog(x, y, style)` | graph with logarithmic axes | arrays → line |

### Module `wavelets_lab.hilbert` (file `src/wavelets_lab/hilbert.py`)

Hilbert-space tools for functions represented by samples on a uniform grid `x_0, x_1, …` with step `h`. The inner product of `L^2` is approximated by the Riemann sum `<f, g> ≈ h * sum_n f[n] * conj(g[n])`; with `h = 1` it is the inner product of `ℓ^2`. A **family** of functions is a 2D array with one sampled function per **row**. Only NumPy is imported.

#### `hilbert.inner_product(f, g, step=1.0)`

**What it is.** The (approximate) inner product `<f, g> = step * sum f[n] conj(g[n])`: linear in `f`, conjugate-linear in `g`.

**Input.**
- `f`, `g` (array-like, same length): samples of two (possibly complex) functions.
- `step` (`float`, default 1): grid step `h`.

**Output.** `complex`.

**How to write it yourself.**
1. `np.vdot(a, b)` computes `sum conj(a) * b`, so `np.vdot(g, f) = sum f * conj(g)`.
2. Return `complex(step * np.vdot(np.asarray(g), np.asarray(f)))`.

#### `hilbert.norm(f, step=1.0)`

**What it is.** The norm `||f|| = sqrt(<f, f>)` induced by the inner product, i.e. the length of the vector or the `L^2` size of the function.

**Input.** `f` (array-like), `step` (`float`).

**Output.** `float` (non-negative).

**How to write it yourself.** `<f, f>` is real and non-negative; return `float(np.sqrt(inner_product(f, f, step).real))`.

#### `hilbert.hat_functions(grid, knots)`

**What it is.** Samples the piecewise linear "hat" functions (splines of order 1): the `k`-th one equals 1 at `knots[k]`, 0 at all other knots, and is linear in between. They form a Riesz basis of the piecewise linear functions that is not orthogonal (neighbouring hats overlap).

**Input.**
- `grid` (array-like): points where the functions are sampled.
- `knots` (array-like): increasing knots.

**Output.** `float64` array of shape `(len(knots), len(grid))`.

**How to write it yourself.**
1. The `k`-th hat is the piecewise linear interpolant of the data "1 at knot `k`, 0 at the other knots", i.e. of the `k`-th row of the identity matrix.
2. `identity = np.eye(len(knots))`.
3. For each row: `np.interp(grid, knots, row)` (linear interpolation).
4. Stack the rows with `np.vstack` and return.

#### `hilbert.gram_matrix(family, step=1.0)`

**What it is.** The Gram matrix `G[i, j] = <e_j, e_i>` of the inner products of all pairs of functions in a family. It is the identity exactly for orthonormal families, and it is invertible exactly for linearly independent ones.

**Input.**
- `family` (array-like): 2D array, one function per row (shape `(n, N)`).
- `step` (`float`).

**Output.** `numpy.ndarray` of shape `(n, n)`, Hermitian (`G = G^*`) and positive semidefinite.

**How to write it yourself.**
1. `rows = np.atleast_2d(np.asarray(family))`.
2. All inner products at once with one matrix product: `step * (rows.conj() @ rows.T)`; entry `[i, j]` is `step * sum conj(e_i) e_j = <e_j, e_i>`.

#### `hilbert.riesz_bounds(family, step=1.0)`

**What it is.** The optimal Riesz bounds `A`, `B` of a finite family: the best constants with `A ||c||^2 <= ||sum_k c_k e_k||^2 <= B ||c||^2` for all coefficient vectors `c`. Since `||sum c_k e_k||^2 = c^* G c`, they are the smallest and largest eigenvalues of the Gram matrix. `A = B = 1` for an orthonormal family; `A > 0` exactly when the family is linearly independent; a ratio `B / A` close to 1 means "almost orthonormal".

**Input.** `family` (array-like), `step` (`float`).

**Output.** Tuple of two floats `(A, B)`.

**How to write it yourself.**
1. Compute the eigenvalues of the Hermitian Gram matrix in increasing order: `np.linalg.eigvalsh(gram_matrix(family, step))`.
2. Return the first and the last one as floats.

#### `hilbert.gram_schmidt(family, step=1.0, tolerance=1e-10)`

**What it is.** The (modified) Gram–Schmidt process: produces an orthonormal family spanning the same space. Functions that are numerically linear combinations of the previous ones are skipped.

**Input.**
- `family` (array-like): one function per row.
- `step` (`float`): grid step for the inner product.
- `tolerance` (`float`): residual norms below this value count as zero.

**Output.** 2D array of orthonormal functions (one per row); it has fewer rows than the input if the family was dependent.

**How to write it yourself.**
1. Convert to a 2D array and to a floating type (`np.result_type(rows.dtype, np.float64)`), start with an empty list `basis`.
2. For every `vector` (row) of the family:
   1. `residual = vector.copy()`.
   2. For every `e` already in `basis`: subtract the projection, `residual = residual - inner_product(residual, e, step) * e`. (Using the current `residual` instead of the original vector is the *modified* variant, which is numerically more stable.)
   3. `length = norm(residual, step)`; if `length > tolerance`, append `residual / length` to `basis`.
3. If `basis` is empty, return an array of shape `(0, N)`; otherwise return `np.vstack(basis)`.

#### `hilbert.is_orthonormal(family, step=1.0, atol=1e-8)`

**What it is.** Checks whether a family is orthonormal, i.e. whether its Gram matrix is the identity.

**Input.** `family` (array-like), `step` (`float`), `atol` (`float`): allowed deviation.

**Output.** `bool`.

**How to write it yourself.**
1. `gram = gram_matrix(family, step)`.
2. Return `bool(np.allclose(gram, np.eye(gram.shape[0]), atol=atol))`.

#### `hilbert.project(f, family, step=1.0)`

**What it is.** The orthogonal projection of `f` onto the span of a family: the closest function to `f` in that space. The residual `f - Pf` is orthogonal to every element of the family.

**Input.** `f` (array-like), `family` (array-like, linearly independent rows), `step` (`float`).

**Output.** Array: samples of the projection (same length as `f`).

**How to write it yourself.** Return `synthesize(expansion_coefficients(f, family, step), family)`.

#### `hilbert.expansion_coefficients(f, family, step=1.0)` (used internally)

**What it is.** The coefficients `c` of the best approximation of `f` by a combination `sum_k c_k e_k` (the orthogonal projection). They solve the normal equations `G c = b` with `b_i = <f, e_i>`. For an orthonormal family `G = I` and simply `c_k = <f, e_k>`; for a Riesz basis this is the expansion in the dual basis.

**Input.**
- `f` (array-like): samples of the function.
- `family` (array-like): linearly independent family, one function per row.
- `step` (`float`).

**Output.** Array of coefficients `c_k` (length = number of rows).

**How to write it yourself.**
1. `rows = np.atleast_2d(np.asarray(family))`.
2. Right-hand side: `b = step * (rows.conj() @ f)` (entry `i` is `<f, e_i>`).
3. Solve the linear system: `np.linalg.solve(gram_matrix(rows, step), b)`.

#### `hilbert.synthesize(coefficients, family)` (used internally)

**What it is.** Builds the linear combination `sum_k c_k e_k` from coefficients.

**Input.** `coefficients` (array-like, length `n`), `family` (array-like, shape `(n, N)`).

**Output.** Array of length `N`: samples of the combination.

**How to write it yourself.** A vector-matrix product does exactly this sum: return `np.asarray(coefficients) @ np.atleast_2d(family)`.